In [4]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('healthcare_data.db')

# Loading and merging
patient_df = pd.read_csv('/content/Patient_Data.csv')
billing_df = pd.read_csv('/content/Billing_Data.csv')
merged_df = pd.merge(patient_df, billing_df, on='PatientID', how='inner')

# Database update
merged_df.to_sql('patient_billing_summary', conn, if_exists='replace', index=False)

# Analysis
summary_stats = merged_df.groupby('Department')['FinalAmount'].describe()
display(summary_stats)

,count,mean,std,min,25%,50%,75%,max
Department,,,,,,,,
Cardiology,3.0,3066.666667,115.470054,3000.0,3000.0,3000.0,3100.0,3200.0
Dermatology,1.0,4000.000000,NaN,4000.0,4000.0,4000.0,4000.0,4000.0
Neurology,1.0,3500.000000,NaN,3500.0,3500.0,3500.0,3500.0,3500.0
Orthopedics,1.0,5000.000000,NaN,5000.0,5000.0,5000.0,5000.0,5000.0


In [7]:
summary_stats.to_csv('department_billing_summary.csv')
conn.close()
print("Summary saved to CSV and database connection closed.")

Summary saved to CSV and database connection closed.


In [8]:
# Further analysis using summary_stats
department_revenue = merged_df.groupby('Department')['FinalAmount'].sum().sort_values(ascending=False)

top_dept = summary_stats['mean'].idxmax()
top_value = summary_stats['mean'].max()

print(f"Department with highest average billing: {top_dept} (${top_value:.2f})")
print("\nTotal revenue by department:")
display(department_revenue)

Department with highest average billing: Orthopedics ($5000.00)

Total revenue by department:


,FinalAmount
Department,
Cardiology,9200
Orthopedics,5000
Dermatology,4000
Neurology,3500


In [13]:
import pandas as pd

# 1. Load patient dataset and show summary
patient_df = pd.read_csv('/content/Patient_Data.csv')
print("--- Task 1: Dataset Summary ---")
patient_df.info()

# 2. Select relevant columns
billing_relevant_df = patient_df[['PatientID', 'Department', 'Doctor', 'BillAmount']].copy()

# 3. Drop administrative columns (already implicitly handled by step 2, but shown for completeness)
# patient_df.drop(columns=['ReceptionistID', 'CheckInTime'], inplace=True, errors='ignore')
print("\n--- Task 2 & 3: Columns Selected ---")
display(billing_relevant_df.head())

--- Task 1: Dataset Summary ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes

--- Task 2 & 3: Columns Selected ---


,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN


In [14]:
# 4. Groupby to find total bill amount per department
dept_total_bill = billing_relevant_df.groupby('Department')['BillAmount'].sum()
print("--- Task 4: Total Bill per Department ---")
display(dept_total_bill)

# 5. Remove duplicate patient records based on PatientID
patient_df_unique = patient_df.drop_duplicates(subset='PatientID').copy()

# 6. Fill missing BillAmount values with the mean
mean_bill = patient_df_unique['BillAmount'].mean()
patient_df_unique['BillAmount'] = patient_df_unique['BillAmount'].fillna(mean_bill)
print(f"\n--- Task 5 & 6: Duplicates removed and missing values filled with {mean_bill:.2f} ---")

--- Task 4: Total Bill per Department ---


,BillAmount
Department,
Cardiology,16200.0
Dermatology,0.0
Neurology,0.0
Orthopedics,7500.0



--- Task 5 & 6: Duplicates removed and missing values filled with 6233.33 ---


In [15]:
# 7. Merge billing dataset with patient dataset on PatientID
billing_df = pd.read_csv('/content/Billing_Data.csv')
merged_data = pd.merge(patient_df_unique, billing_df, on='PatientID', how='inner')

# 8. Concatenate new patients for the current week (row-wise)
# Creating a dummy DataFrame for 'new patients'
new_patients_data = {
    'PatientID': [201, 202],
    'Name': ['John Doe', 'Jane Smith'],
    'Department': ['Cardiology', 'Neurology'],
    'BillAmount': [4500.0, 3800.0]
}
new_patients_df = pd.DataFrame(new_patients_data)
concatenated_rows = pd.concat([merged_data, new_patients_df], axis=0, ignore_index=True)

# 9. Concatenate new billing category columns (column-wise)
# Creating dummy data for specific columns requested
extra_cols = pd.DataFrame({
    'InsuranceCovered': [1500] * len(concatenated_rows),
    'FinalAmount': concatenated_rows['BillAmount'] - 1500
})
final_df = pd.concat([concatenated_rows, extra_cols], axis=1)

print("--- Task 7, 8 & 9: Final Merged and Concatenated Results ---")
display(final_df.tail())

--- Task 7, 8 & 9: Final Merged and Concatenated Results ---


,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime,InsuranceCovered,FinalAmount,InsuranceCovered,FinalAmount
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,1.0,2023-01-12 11:00,2500.0,5000.0,1500,6000.000000
3,104,David,Cardiology,Dr. Smith,6200.000000,3.0,2023-01-13 12:00,3000.0,3200.0,1500,4700.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333,2.0,2023-01-14 08:45,1000.0,4000.0,1500,4733.333333
5,201,John Doe,Cardiology,NaN,4500.000000,NaN,NaN,NaN,NaN,1500,3000.000000
6,202,Jane Smith,Neurology,NaN,3800.000000,NaN,NaN,NaN,NaN,1500,2300.000000
